# layoutlmv3
Original dataset annotations; validation selects the checkpoint.


## 1. Environment
On Kaggle, attach `vhngthnh/reconia-data` and enable the `GH_TOKEN` secret. Locally, set `RECONIA_DATA_DIR` to the raw dataset root. Existing source and outputs are preserved.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

DATASET = "sroie"
GIT_REF = "codex/original-data-six-models"
repo_dir = Path("/kaggle/working/reconia") if Path("/kaggle").exists() else Path.cwd()
if repo_dir.name == "notebooks":
    repo_dir = repo_dir.parent
if not repo_dir.exists():
    import base64
    from kaggle_secrets import UserSecretsClient
    env = os.environ.copy()
    token = UserSecretsClient().get_secret("GH_TOKEN")
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
               GIT_CONFIG_VALUE_0="Authorization: Basic " + auth)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF,
                    "https://github.com/tamir39/reconia.git", str(repo_dir)], env=env, check=True)
    del env, token, auth
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Source:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Prepare the selected dataset


In [ ]:
from importlib import import_module
from utils.common import get_cache_dir

prepare = getattr(import_module(f"prepare.prepare_{DATASET}"), f"prepare_{DATASET}")
prepare()
print("Prepared:", get_cache_dir(DATASET))


## 3. Pretrained weights


In [ ]:
from tools.download_weights import download_weights

download_weights("layoutlmv3")


## 4. Train and validate
Validation runs after each epoch. F1 is the mean of per-field F1 scores.


In [ ]:
from models.layoutlmv3.train import train_model

train_model(DATASET, epochs=30, batch_size=4, accum_steps=2,
            lr=3e-05, min_epochs=5, patience=5, resume=False)


## 5. Validation history


In [ ]:
import pandas as pd

display(pd.read_csv(repo_dir / "results" / DATASET / "layoutlmv3" / "history.csv"))


## 6. Final test (run after experiments are fixed)
Loads the best validation checkpoint; does not update weights.


In [ ]:
RUN_TEST = False
if RUN_TEST:
    train_model(DATASET, batch_size=4, test_only=True)
